# 从 Loss 到第一次参数更新

> 配置文件可以重建模型结构，但刚初始化的参数还没有学到语言规律。一只这样的模型看到 `cats chase` 时，正确答案是 `mice`，它却可能把 9 个 Token 的概率分得接近，也可能把最高概率给了 `sleep`。
>
> 只告诉模型「猜错了」还不够。它需要一个数字说明错了多少，还需要从这个数字得到每个参数应该怎样调整。这个数字就是 **loss**，调整方向来自**反向传播**。
>
> 我们先让一个只有一张参数表的迷你模型学会 `cats chase mice`。等它真的从猜错变成猜对，再加入 batch、padding 和 SFT 的 **label mask**。每一步都会留下概率、曲线或断言作为证据。

句子 `cats chase mice` 可以拆成几道连续的小题：看到 `<bos>` 预测 `cats`，看到 `cats` 预测 `chase`，看到 `chase` 预测 `mice`。

模型刚初始化时还不知道这些对应关系。它可能把 9 个候选 Token 的概率分得接近，也可能把最高分给错误答案。训练首先需要用一个数值衡量预测与正确答案之间的差距，再把这个差距传回参数。

前一个数值称为 **loss**，向后计算参数调整方向的过程称为**反向传播**。下面用一张很小的参数表，完整观察一次预测怎样变成第一次参数更新。

## 0. 语言模型的训练目标

先用一句只有三个普通单词的话：

```text
cats chase mice
```

在句子前后加上 `<bos>` 和 `<eos>`，模型会得到 4 道连续的小题：

| 已经看到的最后一个 token | 应该预测的下一个 token |
|:---|:---|
| `<bos>` | `cats` |
| `cats` | `chase` |
| `chase` | `mice` |
| `mice` | `<eos>` |

**Next-token prediction**：根据前面已经出现的 token，预测下一个 token。通俗地说，就是把一句话变成许多道「下一个词是什么」的小题。

为了把训练过程看得足够清楚，这里先使用 Bigram Language Model。它只根据当前 token 预测下一个 token，不读取更早的上下文。这个模型很小，但 loss、反向传播和参数更新与更大的语言模型遵循相同规则。


In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset

torch.manual_seed(42)

tokens = [
    "<pad>", "<bos>", "<eos>", "cats", "dogs",
    "chase", "mice", "balls", "sleep",
]
token_to_id = {token: index for index, token in enumerate(tokens)}

sentence_tokens = ["<bos>", "cats", "chase", "mice", "<eos>"]
sentence = torch.tensor([token_to_id[token] for token in sentence_tokens])
input_ids = sentence[:-1]
labels = sentence[1:]

target_map = torch.zeros(len(tokens), len(tokens))
target_map[input_ids, labels] = 1

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(target_map, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(tokens)), tokens, rotation=45, ha="right")
ax.set_yticks(range(len(tokens)), tokens)
ax.set_xlabel("Target token")
ax.set_ylabel("Current token")
ax.set_title("Four next-token training pairs")
plt.tight_layout()
plt.show()

assert torch.equal(input_ids, torch.tensor([1, 3, 5, 6]))
assert torch.equal(labels, torch.tensor([3, 5, 6, 2]))


图中的每个蓝色格子就是一道训练题。横轴是答案，纵轴是当前 token。

模型内部也会维护一张 $9\times9$ 的参数表。输入某个 token ID 时，它取出对应的一行，得到对 9 个候选 token 的原始分数。

这些原始分数称为 **logits**。Logits 还不是概率，可以是任意实数；经过 softmax 后，它们才会变成总和为 1 的概率。


In [ ]:
class BigramLanguageModel(nn.Module):
    """只根据当前 token 预测下一个 token 的迷你语言模型。"""

    def __init__(self, vocab_size):
        super().__init__()
        self.transition = nn.Embedding(vocab_size, vocab_size)

    def forward(self, input_ids, attention_mask=None, labels=None):
        """返回 logits；提供 labels 时，同时返回平均 Cross-Entropy Loss。"""
        logits = self.transition(input_ids)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            )
        return {"logits": logits, "loss": loss}


model = BigramLanguageModel(vocab_size=len(tokens))
with torch.no_grad():
    initial_logits = model(input_ids.unsqueeze(0))["logits"][0]
    initial_probs = F.softmax(initial_logits, dim=-1)

fig, ax = plt.subplots(figsize=(8, 3.8))
image = ax.imshow(initial_probs, cmap="YlOrRd", vmin=0, vmax=0.4)
ax.set_xticks(range(len(tokens)), tokens, rotation=45, ha="right")
ax.set_yticks(range(len(input_ids)), sentence_tokens[:-1])
ax.set_xlabel("Predicted next token")
ax.set_ylabel("Current token")
ax.set_title("Probabilities before training")
fig.colorbar(image, ax=ax, label="Probability")
plt.tight_layout()
plt.show()

assert initial_probs.shape == (4, 9)
assert torch.allclose(initial_probs.sum(dim=-1), torch.ones(4))


刚初始化时，每一行的颜色很分散。正确答案不一定是最深的格子，因为参数表还是随机的。

现在缺少的是一个统一的分数：正确 token 的概率越低，分数越大；正确概率越接近 1，分数越接近 0。Cross-Entropy Loss 正好满足这个要求。


## 1. 单个位置的 Cross-Entropy

先只看一道题。假设词表里有 4 个候选 token，模型给出的 logits 是：

$$[2.0,\ 1.0,\ 0.1,\ -1.0]$$

正确答案是第 0 个 token。计算分两步。

第一步，用 softmax 把 logits 转成概率：

$$p_i=\frac{e^{z_i}}{\sum_j e^{z_j}}$$

第二步，只取正确 token 的概率，再计算负对数：

$$L=-\log p_{correct}$$

这就是分类任务里的 Cross-Entropy Loss。对语言模型来说，每个位置都是一次「从整个词表中选出下一个 token」的分类。


In [ ]:
demo_logits = torch.tensor([2.0, 1.0, 0.1, -1.0])
correct_id = 0

exp_values = torch.exp(demo_logits)
demo_probs = exp_values / exp_values.sum()
correct_prob = demo_probs[correct_id].item()
manual_loss = -math.log(correct_prob)
torch_loss = F.cross_entropy(
    demo_logits.unsqueeze(0),
    torch.tensor([correct_id]),
)

colors = ["tab:blue", "lightgray", "lightgray", "lightgray"]
fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(["A", "B", "C", "D"], demo_probs, color=colors)
ax.bar_label(bars, fmt="%.3f")
ax.set_ylim(0, 1)
ax.set_ylabel("Probability")
ax.set_title(f"Correct probability = {correct_prob:.3f}, loss = {manual_loss:.3f}")
plt.tight_layout()
plt.show()

assert abs(manual_loss - torch_loss.item()) < 1e-6


手算结果与 PyTorch 相同。Cross-Entropy 并不关心错误 token 分别叫什么，它只检查正确 token 获得了多少概率。

为什么还要取负对数，而不是直接使用 $1-p_{correct}$？看一条曲线会更直观。


In [ ]:
probability = torch.linspace(0.01, 0.99, 200)
loss_curve = -torch.log(probability)
marked_probs = torch.tensor([0.1, 0.5, 0.9])
marked_losses = -torch.log(marked_probs)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(probability, loss_curve, color="tab:blue")
ax.scatter(marked_probs, marked_losses, color="tab:red", zorder=3)
for prob, loss_value in zip(marked_probs, marked_losses):
    ax.annotate(
        f"p={prob:.1f}, loss={loss_value:.2f}",
        (prob, loss_value),
        xytext=(8, 8),
        textcoords="offset points",
    )
ax.set_xlabel("Probability of the correct token")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Low confidence receives a larger penalty")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


正确概率从 0.9 降到 0.5，loss 从 0.11 增加到 0.69。继续降到 0.1，loss 会增加到 2.30。模型越自信地猜错，受到的惩罚越大。

到这里，loss 只完成了「衡量错误」这一步。训练还需要回答另一个问题：怎样修改 logits，才能让下一次的 loss 更小。


## 2. 单次参数更新

把刚才的 4 个 logits 直接当成可训练参数。反向传播会计算每个参数对 loss 的影响，这个影响称为梯度。

**Gradient**：loss 对某个参数变化的敏感程度。通俗地说，梯度告诉优化器「这个参数往哪个方向移动，loss 会下降」。

下面只执行一次 SGD 更新，然后比较更新前后的概率。


In [ ]:
trainable_logits = nn.Parameter(torch.tensor([2.0, 1.0, 0.1, -1.0]))
optimizer = torch.optim.SGD([trainable_logits], lr=0.5)
target = torch.tensor([correct_id])

before_probs = F.softmax(trainable_logits.detach(), dim=-1)
before_loss = F.cross_entropy(trainable_logits.unsqueeze(0), target)

optimizer.zero_grad()
before_loss.backward()
gradient = trainable_logits.grad.detach().clone()
optimizer.step()

after_probs = F.softmax(trainable_logits.detach(), dim=-1)
after_loss = F.cross_entropy(trainable_logits.unsqueeze(0), target)

x = torch.arange(4)
width = 0.36
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.bar(x - width / 2, before_probs, width, label="Before")
ax.bar(x + width / 2, after_probs, width, label="After")
ax.set_xticks(x, ["A", "B", "C", "D"])
ax.set_ylim(0, 1)
ax.set_ylabel("Probability")
ax.set_title("One gradient step raises the correct probability")
ax.legend()
plt.tight_layout()
plt.show()

assert gradient[correct_id] < 0
assert after_probs[correct_id] > before_probs[correct_id]
assert after_loss < before_loss


正确 token 对应的梯度是负数。SGD 使用 $\theta\leftarrow\theta-\eta g$ 更新参数，因此减去负梯度会提高正确 token 的 logit。其他 token 的概率随之下降。

这一步已经包含训练的核心因果关系：loss 产生梯度，梯度改变参数，参数改变下一次预测。现在把同一个过程扩展到整句话。


## 3. 序列级训练

一句话有 4 个预测位置，因此会得到 4 个位置的 loss。PyTorch 默认把所有有效位置的 loss 取平均：

$$L_{sentence}=\frac{L_1+L_2+L_3+L_4}{4}$$

先观察训练前，哪几道题最难。


In [ ]:
sentence_model = BigramLanguageModel(vocab_size=len(tokens))
sentence_logits = sentence_model(input_ids.unsqueeze(0))["logits"][0]
position_losses = F.cross_entropy(
    sentence_logits,
    labels,
    reduction="none",
)
mean_loss = F.cross_entropy(sentence_logits, labels)

pair_names = [
    f"{current}→{target}"
    for current, target in zip(sentence_tokens[:-1], sentence_tokens[1:])
]
fig, ax = plt.subplots(figsize=(7, 3.8))
bars = ax.bar(pair_names, position_losses.detach(), color="tab:orange")
ax.axhline(mean_loss.item(), color="tab:blue", linestyle="--", label="Mean")
ax.bar_label(bars, fmt="%.2f")
ax.set_ylabel("Loss")
ax.set_title("Loss at each position before training")
ax.tick_params(axis="x", rotation=25)
ax.legend()
plt.tight_layout()
plt.show()

assert torch.allclose(position_losses.mean(), mean_loss)


每个位置的难度不同，但一次 `backward()` 会把平均 loss 对所有相关参数的梯度都算出来。接下来重复执行 80 次更新，并同时记录两个量：平均 loss，以及正确 token 的平均概率。


In [ ]:
optimizer = torch.optim.AdamW(sentence_model.parameters(), lr=0.12)
loss_history = []
correct_prob_history = []

for step in range(80):
    outputs = sentence_model(
        input_ids.unsqueeze(0),
        labels=labels.unsqueeze(0),
    )
    loss = outputs["loss"]

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        probs = F.softmax(outputs["logits"][0], dim=-1)
        correct_probs = probs[torch.arange(len(labels)), labels]
        loss_history.append(loss.item())
        correct_prob_history.append(correct_probs.mean().item())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(loss_history, color="tab:blue")
axes[0].set_title("Training loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.25)

axes[1].plot(correct_prob_history, color="tab:green")
axes[1].set_title("Mean probability of correct tokens")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Probability")
axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

assert loss_history[-1] < loss_history[0] * 0.1
assert correct_prob_history[-1] > 0.9


In [ ]:
with torch.no_grad():
    trained_logits = sentence_model(input_ids.unsqueeze(0))["logits"][0]
    predicted_ids = trained_logits.argmax(dim=-1)

predicted_tokens = [tokens[index] for index in predicted_ids.tolist()]
target_tokens = sentence_tokens[1:]

fig, ax = plt.subplots(figsize=(8, 2.2))
ax.axis("off")
table = ax.table(
    cellText=[sentence_tokens[:-1], target_tokens, predicted_tokens],
    rowLabels=["Current", "Target", "Prediction"],
    colLabels=[f"Position {index}" for index in range(4)],
    cellLoc="center",
    loc="center",
)
table.scale(1, 1.6)
ax.set_title("Predictions after training", pad=16)
plt.tight_layout()
plt.show()

assert predicted_tokens == target_tokens


模型已经把 4 道题全部答对。这里没有额外的人工标签：`labels` 就是原句向左移动一格后的下一个 token。

不过，这个结果只说明模型记住了一句话。真实训练要同时处理许多长度不同的样本，这会引出下一个具体问题。


## 4. 变长序列的 Batch

现在加入第二句话：

```text
cats chase mice   → 4 个预测位置
cats sleep        → 3 个预测位置
```

二维张量的每一行必须相同长度，因此短句需要补上 `<pad>`。但补齐后又产生两个新问题：模型不应该关注 PAD，loss 也不应该把 PAD 当成答案。

**Data Collator**：把多条样本整理成一个 batch 的组件。通俗地说，它负责按当前 batch 的最长样本补齐其他样本，并同时生成 mask。


In [ ]:
def simple_collate(features, pad_id=0, ignore_index=-100):
    """将不同长度的样本补齐为一个 batch。"""
    max_len = max(len(item["input_ids"]) for item in features)
    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []

    for item in features:
        pad_len = max_len - len(item["input_ids"])
        batch_input_ids.append(item["input_ids"] + [pad_id] * pad_len)
        batch_attention_mask.append(
            [1] * len(item["input_ids"]) + [0] * pad_len
        )
        batch_labels.append(
            item["labels"] + [ignore_index] * pad_len
        )

    return {
        "input_ids": torch.tensor(batch_input_ids),
        "attention_mask": torch.tensor(batch_attention_mask),
        "labels": torch.tensor(batch_labels),
    }


short_sentence = ["<bos>", "cats", "sleep", "<eos>"]
short_ids = [token_to_id[token] for token in short_sentence]
features = [
    {"input_ids": sentence[:-1].tolist(), "labels": sentence[1:].tolist()},
    {"input_ids": short_ids[:-1], "labels": short_ids[1:]},
]
batch = simple_collate(features)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
items = [
    ("input_ids", "Input IDs", "Blues"),
    ("attention_mask", "Attention mask", "Greens"),
    ("labels", "Labels", "Oranges"),
]
for ax, (name, title, color_map) in zip(axes, items):
    values = batch[name]
    ax.imshow(values, cmap=color_map, aspect="auto")
    for row in range(values.size(0)):
        for column in range(values.size(1)):
            ax.text(column, row, str(values[row, column].item()), ha="center")
    ax.set_xticks(range(values.size(1)))
    ax.set_yticks([0, 1], ["Long", "Short"])
    ax.set_xlabel("Position")
    ax.set_title(title)
plt.tight_layout()
plt.show()

assert batch["input_ids"].shape == (2, 4)
assert batch["attention_mask"][1, -1].item() == 0
assert batch["labels"][1, -1].item() == -100


三个张量分工不同：

- `input_ids` 用 `<pad>` 的 ID 补齐形状。
- `attention_mask=0` 告诉模型这个位置是 PAD，不应参与注意力计算。
- `labels=-100` 告诉 Cross-Entropy 这个位置不参与 loss。

后两项不能互相替代。`attention_mask` 管模型读取哪些位置，`labels` 管哪些预测要接受评分。


## 5. SFT 的 Label Mask

预训练文本中的每个普通 token 通常都可以作为预测目标。SFT（Supervised Fine-Tuning，监督微调）处理对话时，目标不同：我们主要希望模型学习 assistant 的回答。

**Chat Template**：把带有 `user`、`assistant` 等角色的结构化对话转换成 token 序列的模板。通俗地说，它给不同说话人的内容加上模型认识的角色标记。

看一个已经模板化的最小例子：

```text
完整序列：<user> 2+2? <assistant> 4 <eos>
输入：    <user> 2+2? <assistant> 4
目标：    忽略   忽略  4           <eos>
```

只有 assistant 回答 `4 <eos>` 参与 loss，user 的问题和角色标记使用 `-100` 忽略。


In [ ]:
chat_tokens = ["<user>", "2+2?", "<assistant>", "4", "<eos>"]
chat_vocab = {token: index for index, token in enumerate(chat_tokens)}
chat_input_tokens = chat_tokens[:-1]
chat_labels = torch.tensor([-100, -100, chat_vocab["4"], chat_vocab["<eos>"]])
loss_mask = (chat_labels != -100).float().unsqueeze(0)

fig, ax = plt.subplots(figsize=(7, 1.8))
ax.imshow(loss_mask, cmap="Greens", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(chat_input_tokens)), chat_input_tokens)
ax.set_yticks([])
ax.set_xlabel("Current token position")
ax.set_title("Green positions contribute to SFT loss")
for column, included in enumerate(loss_mask[0]):
    label = "include" if included.item() == 1 else "ignore"
    ax.text(column, 0, label, ha="center", va="center")
plt.tight_layout()
plt.show()

assert chat_labels.tolist() == [-100, -100, 3, 4]


In [ ]:
torch.manual_seed(7)
chat_logits = torch.randn(len(chat_input_tokens), len(chat_vocab))
valid_positions = chat_labels != -100
valid_losses = F.cross_entropy(
    chat_logits[valid_positions],
    chat_labels[valid_positions],
    reduction="none",
)
display_losses = torch.zeros(len(chat_input_tokens))
display_losses[valid_positions] = valid_losses.detach()
sft_loss = valid_losses.mean()

colors = ["lightgray" if not valid else "tab:green" for valid in valid_positions]
fig, ax = plt.subplots(figsize=(7, 3.4))
bars = ax.bar(chat_input_tokens, display_losses, color=colors)
for bar, valid, value in zip(bars, valid_positions, display_losses):
    label = f"{value.item():.2f}" if valid else "ignored"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), label, ha="center")
ax.set_ylabel("Token loss")
ax.set_title(f"SFT averages only green positions: loss = {sft_loss:.2f}")
plt.tight_layout()
plt.show()

assert valid_positions.sum().item() == 2
assert torch.allclose(sft_loss, valid_losses.mean())


灰色位置虽然经过模型并产生 logits，但不进入最终平均 loss。这样，梯度主要来自 assistant 的回答。

需要注意，是否遮住 user 和 system 部分取决于训练目标与具体 recipe。这里展示的是常见的 assistant-only SFT；如果希望模型同时学习完整对话格式，也可以让更多位置参与 loss。


## 6. 完整训练循环

前面分别验证了单句 loss、参数更新和 batch。现在用两类句子训练同一个模型：

```text
cats chase mice
dogs chase balls
```

Dataset 负责返回单条训练样本，DataLoader 和 collator 负责组成 batch。每个 code cell 仍然保留一个可检查的结果。


In [ ]:
class ToyTextDataset(Dataset):
    """把完整 token 序列转换成 input_ids 与右移一格的 labels。"""

    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        ids = self.sequences[index]
        return {"input_ids": ids[:-1], "labels": ids[1:]}


cats_sequence = ["<bos>", "cats", "chase", "mice", "<eos>"]
dogs_sequence = ["<bos>", "dogs", "chase", "balls", "<eos>"]
training_sequences = []
for sequence_tokens in [cats_sequence, dogs_sequence] * 4:
    training_sequences.append([token_to_id[token] for token in sequence_tokens])

dataset = ToyTextDataset(training_sequences)
sequence_lengths = [len(item) - 1 for item in training_sequences]

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar(range(len(sequence_lengths)), sequence_lengths, color="tab:purple")
ax.set_xticks(range(len(sequence_lengths)))
ax.set_xlabel("Sample index")
ax.set_ylabel("Prediction positions")
ax.set_title("Eight training samples")
ax.set_ylim(0, 5)
plt.tight_layout()
plt.show()

assert len(dataset) == 8


In [ ]:
torch.manual_seed(42)
dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=simple_collate,
)
batch_model = BigramLanguageModel(vocab_size=len(tokens))
batch_optimizer = torch.optim.AdamW(batch_model.parameters(), lr=0.08)
total_steps = 30 * len(dataloader)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    batch_optimizer,
    lr_lambda=lambda step: max(0.1, 1 - step / total_steps),
)

step_losses = []
learning_rates = []
gradient_norms = []

for epoch in range(30):
    for batch in dataloader:
        outputs = batch_model(**batch)
        loss = outputs["loss"]

        batch_optimizer.zero_grad()
        loss.backward()
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            batch_model.parameters(),
            max_norm=1.0,
        )
        batch_optimizer.step()
        scheduler.step()

        step_losses.append(loss.item())
        gradient_norms.append(gradient_norm.item())
        learning_rates.append(scheduler.get_last_lr()[0])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(step_losses, color="tab:blue")
axes[0].set_title("Batch loss")
axes[0].set_xlabel("Optimizer step")
axes[0].set_ylabel("Loss")

axes[1].plot(gradient_norms, color="tab:orange")
axes[1].set_title("Gradient norm")
axes[1].set_xlabel("Optimizer step")
axes[1].set_ylabel("Norm")

axes[2].plot(learning_rates, color="tab:green")
axes[2].set_title("Learning rate")
axes[2].set_xlabel("Optimizer step")
axes[2].set_ylabel("LR")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

assert step_losses[-1] < step_losses[0]
assert learning_rates[-1] < learning_rates[0]


这一次，训练 loop 里的每一行都有可观察的作用：

- `model(**batch)` 产生 logits 和 loss。
- `loss.backward()` 产生梯度，右图中间记录了梯度范数。
- `optimizer.step()` 根据梯度更新参数，左图的 loss 因此下降。
- `scheduler.step()` 调整后续更新使用的学习率，右图展示了变化。

Trainer 的核心工作就是稳定地重复这些步骤，并在外面加入梯度累积、混合精度、评估和 checkpoint。现在再看到 `trainer.train()`，每一层封装都能对应到已经运行过的操作。


## 7. 训练结果检查

训练 loss 下降不等于模型已经理解语言。我们的 Bigram 模型只看当前 token。训练数据里出现了两道题：

```text
cats chase → mice
dogs chase → balls
```

预测目标都位于 `chase` 后面，但 Bigram 模型看不到更早的 `cats` 或 `dogs`。它只能学到「`chase` 后面一半是 `mice`，一半是 `balls`」。下面直接检查这个稳定失败。


In [ ]:
chase_id = token_to_id["chase"]
with torch.no_grad():
    chase_logits = batch_model(torch.tensor([chase_id]))["logits"][0]
    chase_probs = F.softmax(chase_logits, dim=-1)

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = [
    "tab:orange" if token in {"mice", "balls"} else "lightgray"
    for token in tokens
]
bars = ax.bar(tokens, chase_probs, color=colors)
ax.bar_label(bars, fmt="%.2f")
ax.set_ylim(0, 1)
ax.set_ylabel("Probability")
ax.set_title("Bigram model cannot use earlier context")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

mice_prob = chase_probs[token_to_id["mice"]].item()
balls_prob = chase_probs[token_to_id["balls"]].item()
assert abs(mice_prob - balls_prob) < 0.12
assert mice_prob + balls_prob > 0.85


这个失败不是优化器没有工作，而是模型输入中缺少完成任务所需的信息。Transformer 使用更长的上下文后，`cats` 和 `dogs` 才能影响 `chase` 后面的预测。

因此，读训练曲线时至少要分清两类问题：loss 是否在下降，模型结构是否有能力表示任务需要的规律。前者检查训练过程，后者检查模型能力。


## 8. 工业训练库对照

到这里再映射到 Hugging Face Transformers 或 ModelScope `ms-swift`，对应关系会简单很多：

| 本章已经运行的对象 | 工业训练库中的常见对象 |
|:---|:---|
| `ToyTextDataset` | `Dataset` 或训练数据集封装 |
| `simple_collate` | Data Collator |
| `BigramLanguageModel` | `AutoModelForCausalLM` 等模型 |
| `loss.backward()` | Trainer 内部的 backward |
| `AdamW.step()` | Trainer 创建或接收的 optimizer |
| `scheduler.step()` | learning-rate scheduler |
| loss、梯度范数曲线 | logging 与监控 |

工业库改变的是规模和工程实现，不会改变本章验证过的因果链：较低的正确 token 概率产生较大的 loss，loss 通过梯度改变参数，新的参数再改变预测概率。


## 小结

请确认下面这些问题已经能够独立回答：

- [ ] 我能把一句话拆成多组 next-token prediction 训练对。
- [ ] 我能从 logits 手算 softmax、正确 token 概率和 Cross-Entropy Loss。
- [ ] 我知道一次 `backward()` 和 `optimizer.step()` 怎样提高正确 token 的概率。
- [ ] 我能解释句子 loss 为什么是所有有效 token loss 的平均值。
- [ ] 我能区分 `attention_mask=0` 与 `labels=-100` 的作用。
- [ ] 我知道 assistant-only SFT 为什么要遮住 user 和 template 位置。
- [ ] 我能从 loss、梯度范数和学习率曲线检查训练 loop。
- [ ] 我知道 loss 下降不能证明模型结构已经具备任务需要的能力。


## 作业

> 可以用 AI 询问思路、拆分步骤或检查方向，但不建议直接让 AI 完成整道题。

### 作业 1：从正确概率计算 loss

某个位置上，模型给正确 token 的概率是 0.25。请计算这个位置的 Cross-Entropy Loss。

小提示：使用 $-\log(p)$，Python 中可以调用 `math.log`。


In [ ]:
correct_probability = 0.25

# TODO：把三引号里的内容替换成 loss 计算式
exercise_loss = """在这里计算 -log(correct_probability)"""

assert not isinstance(exercise_loss, str), "请先替换占位内容"
assert abs(exercise_loss - 1.386294) < 1e-5, exercise_loss
print("作业 1 通过：你能把正确 token 的概率换算成 Cross-Entropy Loss。")


### 作业 2：为短句补齐 labels

一条短句有 3 个有效 label，同 batch 的最长序列有 5 个位置。请补出完整 labels。

小提示：PAD 位置不参与 loss，要使用 `-100`。


In [ ]:
valid_labels = [3, 5, 2]
max_length = 5

# TODO：把三引号里的内容替换成补齐后的列表
padded_labels = """在这里用 -100 把 valid_labels 补到 max_length"""

assert not isinstance(padded_labels, str), "请先替换占位内容"
assert padded_labels == [3, 5, 2, -100, -100], padded_labels
print("作业 2 通过：你能用 -100 排除 PAD 位置的 loss。")


### 作业 3：完成一次参数更新

下面已经准备好一个只有 3 个 logits 的参数。补上反向传播与参数更新，让正确 token 的概率提高。

小提示：依次调用 `loss.backward()` 和 `optimizer.step()`。


In [ ]:
exercise_logits = nn.Parameter(torch.tensor([0.2, 0.1, -0.3]))
exercise_optimizer = torch.optim.SGD([exercise_logits], lr=0.5)
exercise_target = torch.tensor([1])
before = F.softmax(exercise_logits.detach(), dim=-1)[1].item()
exercise_loss = F.cross_entropy(exercise_logits.unsqueeze(0), exercise_target)

exercise_optimizer.zero_grad()
# TODO：在这里补上两行：反向传播，然后更新参数

after = F.softmax(exercise_logits.detach(), dim=-1)[1].item()
assert after > before, f"正确 token 的概率没有提高：{before:.3f} -> {after:.3f}"
print("作业 3 通过：你已经独立完成了一次 loss → gradient → update。")


## 参考资料

- [PyTorch CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)：核对 logits、target 与 `ignore_index` 的正式定义。
- [PyTorch autograd](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)：解释计算图、梯度与 `backward()`。
- [PyTorch AdamW](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html)：查看本章训练 loop 使用的优化器接口。
- [Hugging Face causal language modeling](https://huggingface.co/docs/transformers/tasks/language_modeling)：对照真实 Causal LM 的数据整理与训练入口。
